In [1]:
import pandas as pd
import numpy as np

app_train = pd.read_csv('../data/processed/application_train_cleaned.csv')
bureau = pd.read_csv('../data/raw/bureau.csv')
bureau_balance = pd.read_csv('../data/raw/bureau_balance.csv')

print("app_train:", app_train.shape)
print("bureau:", bureau.shape)
print("bureau_balance:", bureau_balance.shape)

app_train: (307507, 313)
bureau: (1716428, 17)
bureau_balance: (27299925, 3)


In [2]:
# Map STATUS to a numeric severity scale
# 0 = current/no DPD, 1-5 = increasing overdue severity, C = closed, X = unknown
status_map = {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, 'C': 0, 'X': 0}
bureau_balance['STATUS_NUM'] = bureau_balance['STATUS'].map(status_map)

bureau_balance_agg = bureau_balance.groupby('SK_ID_BUREAU').agg(
    BUREAU_BALANCE_WORST_STATUS=('STATUS_NUM', 'max'),
    BUREAU_BALANCE_MONTHS_COUNT=('MONTHS_BALANCE', 'count')
).reset_index()

print(bureau_balance_agg.shape)
bureau_balance_agg.head()

(817395, 3)


,SK_ID_BUREAU,BUREAU_BALANCE_WORST_STATUS,BUREAU_BALANCE_MONTHS_COUNT
0,5001709,0,97
1,5001710,0,83
2,5001711,0,4
3,5001712,0,19
4,5001713,0,22


In [3]:
bureau = bureau.merge(bureau_balance_agg, on='SK_ID_BUREAU', how='left')

bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    BUREAU_COUNT=('SK_ID_BUREAU', 'count'),
    BUREAU_ACTIVE_COUNT=('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
    BUREAU_OVERDUE_COUNT=('CREDIT_DAY_OVERDUE', lambda x: (x > 0).sum()),
    BUREAU_AVG_DAYS_CREDIT=('DAYS_CREDIT', 'mean'),
    BUREAU_MAX_OVERDUE_AMT=('AMT_CREDIT_MAX_OVERDUE', 'max'),
    BUREAU_CREDIT_TYPE_DIVERSITY=('CREDIT_TYPE', 'nunique'),
    BUREAU_BALANCE_WORST_STATUS=('BUREAU_BALANCE_WORST_STATUS', 'max')
).reset_index()

print("Aggregated bureau features:", bureau_agg.shape)
bureau_agg.head()

Aggregated bureau features: (305811, 8)


,SK_ID_CURR,BUREAU_COUNT,BUREAU_ACTIVE_COUNT,BUREAU_OVERDUE_COUNT,BUREAU_AVG_DAYS_CREDIT,BUREAU_MAX_OVERDUE_AMT,BUREAU_CREDIT_TYPE_DIVERSITY,BUREAU_BALANCE_WORST_STATUS
0,100001,7,3,0,-735.000000,NaN,1,1.0
1,100002,8,2,0,-874.000000,5043.645,2,1.0
2,100003,4,1,0,-1400.750000,0.000,2,NaN
3,100004,2,0,0,-867.000000,0.000,1,NaN
4,100005,3,2,0,-190.666667,0.000,2,0.0


In [4]:
app_train_fe = app_train.merge(bureau_agg, on='SK_ID_CURR', how='left')

print("Before merge:", app_train.shape)
print("After merge:", app_train_fe.shape)

# Check how many applicants had no bureau history at all
print("Applicants with no bureau record:", app_train_fe['BUREAU_COUNT'].isnull().sum())

Before merge: (307507, 313)
After merge: (307507, 320)
Applicants with no bureau record: 44020


In [5]:
#No-bureau-history case
app_train_fe['NO_BUREAU_HISTORY'] = app_train_fe['BUREAU_COUNT'].isnull().astype(int)
app_train_fe['BUREAU_COUNT'] = app_train_fe['BUREAU_COUNT'].fillna(0)
app_train_fe['BUREAU_ACTIVE_COUNT'] = app_train_fe['BUREAU_ACTIVE_COUNT'].fillna(0)
app_train_fe['BUREAU_OVERDUE_COUNT'] = app_train_fe['BUREAU_OVERDUE_COUNT'].fillna(0)
# Leave the rate/average/max features as NaN — filling with 0 would be misleading
# (0 overdue amount implies "no debt" not "no data"). Tree models handle NaN fine.

print(app_train_fe['NO_BUREAU_HISTORY'].value_counts())

NO_BUREAU_HISTORY
0    263487
1     44020
Name: count, dtype: int64


C:\Users\User\AppData\Local\Temp\ipykernel_31464\2546029675.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app_train_fe['NO_BUREAU_HISTORY'] = app_train_fe['BUREAU_COUNT'].isnull().astype(int)


In [6]:
previous_app = pd.read_csv('../data/raw/previous_application.csv')
print("previous_application:", previous_app.shape)
previous_app['NAME_CONTRACT_STATUS'].value_counts()

previous_application: (1670214, 37)


NAME_CONTRACT_STATUS
Approved        1036781
Canceled         316319
Refused          290678
Unused offer      26436
Name: count, dtype: int64

In [7]:
prev_agg = previous_app.groupby('SK_ID_CURR').agg(
    PREV_APP_COUNT=('SK_ID_PREV', 'count'),
    PREV_APPROVED_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').sum()),
    PREV_REFUSED_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').sum()),
    PREV_CANCELLED_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Canceled').sum()),
    PREV_AVG_CREDIT_AMT=('AMT_CREDIT', 'mean')
).reset_index()

# Approval rate — guard against division by zero
prev_agg['PREV_APPROVAL_RATE'] = prev_agg['PREV_APPROVED_COUNT'] / prev_agg['PREV_APP_COUNT']

print(prev_agg.shape)
prev_agg.head()

(338857, 7)


,SK_ID_CURR,PREV_APP_COUNT,PREV_APPROVED_COUNT,PREV_REFUSED_COUNT,PREV_CANCELLED_COUNT,PREV_AVG_CREDIT_AMT,PREV_APPROVAL_RATE
0,100001,1,1,0,0,23787.00,1.0
1,100002,1,1,0,0,179055.00,1.0
2,100003,3,3,0,0,484191.00,1.0
3,100004,1,1,0,0,20106.00,1.0
4,100005,2,1,0,1,20076.75,0.5


In [8]:
app_train_fe = app_train_fe.merge(prev_agg, on='SK_ID_CURR', how='left')

print("Shape after previous_application merge:", app_train_fe.shape)
print("Applicants with no previous application history:", app_train_fe['PREV_APP_COUNT'].isnull().sum())

Shape after previous_application merge: (307507, 327)
Applicants with no previous application history: 16454


In [9]:
app_train_fe['NO_PREV_APP_HISTORY'] = app_train_fe['PREV_APP_COUNT'].isnull().astype(int)

for col in ['PREV_APP_COUNT', 'PREV_APPROVED_COUNT', 'PREV_REFUSED_COUNT', 'PREV_CANCELLED_COUNT']:
    app_train_fe[col] = app_train_fe[col].fillna(0)

# PREV_APPROVAL_RATE and PREV_AVG_CREDIT_AMT stay NaN when there's no history — 
# filling with 0 would falsely suggest "0% approval rate" for brand-new applicants
print(app_train_fe['NO_PREV_APP_HISTORY'].value_counts())

NO_PREV_APP_HISTORY
0    291053
1     16454
Name: count, dtype: int64


C:\Users\User\AppData\Local\Temp\ipykernel_31464\1014098763.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app_train_fe['NO_PREV_APP_HISTORY'] = app_train_fe['PREV_APP_COUNT'].isnull().astype(int)


In [10]:
installments = pd.read_csv('../data/raw/installments_payments.csv')
print("installments_payments:", installments.shape)
installments.head()

installments_payments: (13605401, 8)


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585


In [11]:
# Positive DAYS_LATE means paid late; negative/zero means on-time or early
installments['DAYS_LATE'] = installments['DAYS_ENTRY_PAYMENT'] - installments['DAYS_INSTALMENT']
installments['IS_LATE'] = (installments['DAYS_LATE'] > 0).astype(int)

# Positive UNDERPAYMENT means paid less than scheduled
installments['UNDERPAYMENT'] = installments['AMT_INSTALMENT'] - installments['AMT_PAYMENT']
installments['IS_UNDERPAID'] = (installments['UNDERPAYMENT'] > 0).astype(int)

installments[['DAYS_LATE', 'IS_LATE', 'UNDERPAYMENT', 'IS_UNDERPAID']].describe()

,DAYS_LATE,IS_LATE,UNDERPAYMENT,IS_UNDERPAID
count,1.360250e+07,1.360540e+07,1.360250e+07,1.360540e+07
mean,-8.787405e+00,8.428043e-02,-1.871538e+02,9.521902e-02
std,2.553990e+01,2.778079e-01,1.910673e+04,2.935172e-01
min,-3.189000e+03,0.000000e+00,-2.630909e+06,0.000000e+00
25%,-1.400000e+01,0.000000e+00,0.000000e+00,0.000000e+00
50%,-6.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,2.884000e+03,1.000000e+00,2.424726e+06,1.000000e+00


In [12]:
installments_agg = installments.groupby('SK_ID_CURR').agg(
    INSTALLMENT_LATE_COUNT=('IS_LATE', 'sum'),
    INSTALLMENT_TOTAL_COUNT=('IS_LATE', 'count'),
    INSTALLMENT_AVG_DAYS_LATE=('DAYS_LATE', 'mean'),
    INSTALLMENT_STD_DAYS_LATE=('DAYS_LATE', 'std'),
    INSTALLMENT_UNDERPAYMENT_COUNT=('IS_UNDERPAID', 'sum')
).reset_index()

installments_agg['INSTALLMENT_LATE_RATIO'] = (
    installments_agg['INSTALLMENT_LATE_COUNT'] / installments_agg['INSTALLMENT_TOTAL_COUNT']
)

print(installments_agg.shape)
installments_agg.head()

(339587, 7)


,SK_ID_CURR,INSTALLMENT_LATE_COUNT,INSTALLMENT_TOTAL_COUNT,INSTALLMENT_AVG_DAYS_LATE,INSTALLMENT_STD_DAYS_LATE,INSTALLMENT_UNDERPAYMENT_COUNT,INSTALLMENT_LATE_RATIO
0,100001,1,7,-7.285714,14.625483,0,0.142857
1,100002,0,19,-20.421053,4.925171,0,0.000000
2,100003,0,25,-7.160000,3.726929,0,0.000000
3,100004,0,3,-7.666667,4.163332,0,0.000000
4,100005,1,9,-23.555556,13.510284,0,0.111111


In [13]:
app_train_fe = app_train_fe.merge(installments_agg, on='SK_ID_CURR', how='left')

app_train_fe['NO_INSTALLMENT_HISTORY'] = app_train_fe['INSTALLMENT_TOTAL_COUNT'].isnull().astype(int)

for col in ['INSTALLMENT_LATE_COUNT', 'INSTALLMENT_TOTAL_COUNT', 'INSTALLMENT_UNDERPAYMENT_COUNT']:
    app_train_fe[col] = app_train_fe[col].fillna(0)
# Ratios/averages stay NaN when there's no payment history — same logic as before

print("Shape after installments merge:", app_train_fe.shape)
print("Applicants with no installment history:", app_train_fe['NO_INSTALLMENT_HISTORY'].sum())

Shape after installments merge: (307507, 335)
Applicants with no installment history: 15868


C:\Users\User\AppData\Local\Temp\ipykernel_31464\69182018.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app_train_fe['NO_INSTALLMENT_HISTORY'] = app_train_fe['INSTALLMENT_TOTAL_COUNT'].isnull().astype(int)


In [14]:
pos_cash = pd.read_csv('../data/raw/POS_CASH_balance.csv')
credit_card = pd.read_csv('../data/raw/credit_card_balance.csv')

print("POS_CASH_balance:", pos_cash.shape)
print("credit_card_balance:", credit_card.shape)

POS_CASH_balance: (10001358, 8)
credit_card_balance: (3840312, 23)


In [15]:
pos_cash_agg = pos_cash.groupby('SK_ID_CURR').agg(
    POS_DPD_MAX=('SK_DPD', 'max'),
    POS_DPD_DEF_MAX=('SK_DPD_DEF', 'max'),
    POS_MONTHS_COUNT=('MONTHS_BALANCE', 'count')
).reset_index()

print(pos_cash_agg.shape)
pos_cash_agg.head()

(337252, 4)


,SK_ID_CURR,POS_DPD_MAX,POS_DPD_DEF_MAX,POS_MONTHS_COUNT
0,100001,7,7,9
1,100002,0,0,19
2,100003,0,0,28
3,100004,0,0,4
4,100005,0,0,11


In [16]:
# Utilization = balance relative to credit limit (guard against divide-by-zero)
credit_card['UTILIZATION'] = credit_card['AMT_BALANCE'] / credit_card['AMT_CREDIT_LIMIT_ACTUAL'].replace(0, np.nan)

# Flag months where only minimum payment was made
credit_card['MIN_PAYMENT_ONLY'] = (
    credit_card['AMT_PAYMENT_CURRENT'] <= credit_card['AMT_INST_MIN_REGULARITY']
).astype(int)

credit_card_agg = credit_card.groupby('SK_ID_CURR').agg(
    CC_UTILIZATION_AVG=('UTILIZATION', 'mean'),
    CC_MIN_PAYMENT_ONLY_COUNT=('MIN_PAYMENT_ONLY', 'sum'),
    CC_MONTHS_COUNT=('MONTHS_BALANCE', 'count')
).reset_index()

print(credit_card_agg.shape)
credit_card_agg.head()

(103558, 4)


,SK_ID_CURR,CC_UTILIZATION_AVG,CC_MIN_PAYMENT_ONLY_COUNT,CC_MONTHS_COUNT
0,100006,0.000000,0,6
1,100011,0.302678,31,74
2,100013,0.115301,1,96
3,100021,0.000000,0,17
4,100023,0.000000,0,8


In [17]:
app_train_fe = app_train_fe.merge(pos_cash_agg, on='SK_ID_CURR', how='left')
app_train_fe = app_train_fe.merge(credit_card_agg, on='SK_ID_CURR', how='left')

app_train_fe['NO_POS_CASH_HISTORY'] = app_train_fe['POS_MONTHS_COUNT'].isnull().astype(int)
app_train_fe['NO_CREDIT_CARD_HISTORY'] = app_train_fe['CC_MONTHS_COUNT'].isnull().astype(int)

print("Shape after all merges:", app_train_fe.shape)

Shape after all merges: (307507, 343)


C:\Users\User\AppData\Local\Temp\ipykernel_31464\541078390.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app_train_fe['NO_POS_CASH_HISTORY'] = app_train_fe['POS_MONTHS_COUNT'].isnull().astype(int)
C:\Users\User\AppData\Local\Temp\ipykernel_31464\541078390.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app_train_fe['NO_CREDIT_CARD_HISTORY'] = app_train_fe['CC_MONTHS_COUNT'].isnull().astype(int)


In [18]:
app_train_fe['DEBT_TO_INCOME_RATIO'] = app_train_fe['AMT_CREDIT'] / app_train_fe['AMT_INCOME_TOTAL_CAPPED']
app_train_fe['ANNUITY_TO_INCOME_RATIO'] = app_train_fe['AMT_ANNUITY'] / app_train_fe['AMT_INCOME_TOTAL_CAPPED']
app_train_fe['CREDIT_TERM'] = app_train_fe['AMT_CREDIT'] / app_train_fe['AMT_ANNUITY']
app_train_fe['EMPLOYMENT_TO_AGE_RATIO'] = app_train_fe['DAYS_EMPLOYED'] / app_train_fe['DAYS_BIRTH']

app_train_fe[['DEBT_TO_INCOME_RATIO', 'ANNUITY_TO_INCOME_RATIO', 'CREDIT_TERM', 'EMPLOYMENT_TO_AGE_RATIO']].describe()

C:\Users\User\AppData\Local\Temp\ipykernel_31464\9404836.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app_train_fe['DEBT_TO_INCOME_RATIO'] = app_train_fe['AMT_CREDIT'] / app_train_fe['AMT_INCOME_TOTAL_CAPPED']
C:\Users\User\AppData\Local\Temp\ipykernel_31464\9404836.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app_train_fe['ANNUITY_TO_INCOME_RATIO'] = app_train_fe['AMT_ANNUITY'] / app_train_fe['AMT_INCOME_TOTAL_CAPPED']
C:\Users\User\AppData\Local\Temp\ipykernel_31464\9404836.py:3: PerformanceWarning: DataFrame is 

,DEBT_TO_INCOME_RATIO,ANNUITY_TO_INCOME_RATIO,CREDIT_TERM,EMPLOYMENT_TO_AGE_RATIO
count,307507.000000,307495.000000,307495.000000,252133.000000
mean,3.959393,0.181021,21.612343,0.156860
std,2.688404,0.094504,7.823872,0.133548
min,0.083333,0.008700,8.036674,-0.000000
25%,2.021055,0.114820,15.614473,0.056098
50%,3.267273,0.162880,20.000000,0.118733
75%,5.160000,0.229080,27.099985,0.219167
max,84.736842,1.875965,45.305079,0.728811


In [19]:
print("Final shape:", app_train_fe.shape)
app_train_fe.to_csv('../data/processed/application_train_features.csv', index=False)
print("Saved to data/processed/application_train_features.csv")

Final shape: (307507, 347)
Saved to data/processed/application_train_features.csv
